**Latar Belakang**

1. Latar Belakang Masalah Bisnis

Dalam dunia bisnis, data transaksi terus meningkat seiring bertambahnya aktivitas pelanggan. Namun, data sering mengandung kesalahan seperti harga negatif, tanggal tidak valid, data kosong, dan duplikasi. Hal ini dapat menyebabkan hasil analisis bisnis menjadi tidak akurat. Oleh karena itu, diperlukan proses data cleansing untuk menjaga kualitas data.

2. Target Analisis

Analisis ini bertujuan untuk mengidentifikasi dan membersihkan anomali menggunakan PySpark. Selain itu, dilakukan perbandingan tiga format penyimpanan berdasarkan ukuran file dan kecepatan pembacaan untuk mengetahui format yang paling efisien.

3. Skema 3 Dataset

Proyek ini menggunakan satu dataset utama, yaitu data transaksi dengan sekitar 5% - 10 % anomali. Setelah dibersihkan, data disimpan dalam tiga format:

* CSV: Format data berbentuk tabel.
* JSONL: Format JSON dengan satu objek per baris.
* Parquet: Format kolumnar yang dioptimalkan untuk analisis Big Data.

In [1]:
%pip install pyspark

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: C:\Users\Wilson Kohar\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


# **Cell 1: Inisialisasi Spark Session & Library**
Setiap pemrosesan Big Data dengan PySpark harus dimulai dengan membuat SparkSession. Ini adalah "mesin" yang akan mengeksekusi kode Anda secara paralel.

In [2]:
# CELL 1
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, lower,count,isnan
import time
import os

# Memulai sesi PySpark
spark = SparkSession.builder \
    .appName("Milestone1_Cleansing_Benchmark") \
    .master("local[*]") \
    .getOrCreate()

print("Spark Session berhasil dibuat!")

C:\Users\Wilson Kohar\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Spark Session berhasil dibuat!


# **Cell 2: Memuat Data Transaksi yang Kotor**
Kita akan memuat file transactions.csv yang sebelumnya sudah diinjeksi 5% anomali, lalu melihat total baris awalnya

In [3]:
# CELL 2
# Membaca data CSV
df_raw = spark.read.csv("data_generator/transactions.csv", header=True, inferSchema=True)

print(f"Total baris sebelum cleansing: {df_raw.count():,}")
df_raw.show(5)

Total baris sebelum cleansing: 1,050,000
+--------------+-----------+-------------------+----------+-----------+--------+-------+--------+--------------+--------+
|transaction_id|customer_id|          timestamp|product_id|   category|quantity|  price|discount|payment_method|    city|
+--------------+-----------+-------------------+----------+-----------+--------+-------+--------+--------------+--------+
|  TXN-5C3CA7C0| CUST-35490|2023-09-30 21:59:00|   PRD-083|     Sports|      13|3427034|    0.49|      E-Wallet| Jakarta|
|  TXN-4EFD8E17| CUST-20875|2023-07-12 22:28:00|   PRD-231|       Home|       7|4700598|    0.39|           COD| Jakarta|
|  TXN-F5B7E400| CUST-35559|2023-11-26 01:15:00|   PRD-274|Electronics|      11|3374150|    0.36|      E-Wallet| Jakarta|
|  TXN-A750F9C4| CUST-28754|2023-07-21 21:47:00|   PRD-362|       Home|       8|3947997|    0.31|      Transfer|Surabaya|
|  TXN-9D5E9DF6| CUST-05948|2024-07-01 15:50:00|   PRD-094|Electronics|      14| 694066|    0.24|        

In [4]:
#Part 1
from pyspark.sql.functions import col

print("Distribusi penulisan nama kota pada data mentah:")
# Mengelompokkan data berdasarkan kota dan menghitung jumlahnya
df_raw.groupBy("city").count().orderBy(col("count").desc()).show()

Distribusi penulisan nama kota pada data mentah:
+-----------+------+
|       city| count|
+-----------+------+
|    Jakarta|251122|
|    Bandung|250119|
|   Surabaya|249415|
|      Medan|249344|
|DKI Jakarta| 16753|
|    jakarta| 16646|
|        JKT| 16601|
+-----------+------+



In [5]:

print("--- 2. PROFILING HARGA (Mencari Karakter/Nilai Minus) ---")
# Mengurutkan harga dari yang paling kecil (ascending) untuk melihat nilai negatif di urutan teratas
df_raw.orderBy(col("price").asc()).select("transaction_id", "category", "price").show(5)

print("--- 3. PROFILING TANGGAL (Mencari Tanggal Masa Depan) ---")
# Mengurutkan tanggal dari yang paling baru (descending) untuk melihat anomali tahun 2099
df_raw.orderBy(col("timestamp").desc()).select("transaction_id", "timestamp", "price").show(5)

print("--- 4. PREVIEW SELURUH FIELD UNTUK MISSING VALUES (Null) ---")
# Memindai seluruh kolom untuk menghitung berapa banyak baris yang kosong (null/nan)

# Ini akan membuktikan ada data kosong di customer_id, timestamp, dan category
df_raw.select([
    count(when(col(c).isNull(),c)).alias(c) for c in df_raw.columns]).show()

from pyspark.sql.functions import col

print("--- 5. PROFILING DUPLIKASI PER FIELD ---")

# Melakukan perulangan untuk mengecek setiap kolom di dalam dataset
for column_name in df_raw.columns:
    # Mengelompokkan data berdasarkan kolom saat ini, hitung, dan saring yang > 1
    df_duplicates = df_raw.groupBy(column_name).count().filter(col("count") > 1)

    # Menghitung ada berapa banyak macam data yang berulang
    total_variasi_duplikat = df_duplicates.count()

    print(f"\n=========================================")
    print(f"Field: '{column_name}'")
    print(f"Total macam data yang duplikat: {total_variasi_duplikat:,}")
    print(f"=========================================")

    if total_variasi_duplikat > 0:
        print("Contoh karakter data yang terduplikasi:")
        # Menampilkan data yang paling banyak diduplikasi di urutan atas
        df_duplicates.orderBy(col("count").desc()).show(10, truncate=False)
    else:
        print("Status: Bersih (Semua data unik)")


--- 2. PROFILING HARGA (Mencari Karakter/Nilai Minus) ---
+--------------+-----------+-------+
|transaction_id|   category|  price|
+--------------+-----------+-------+
|  TXN-28283088|       Home|-250000|
|  TXN-F2A05845|Electronics|-250000|
|  TXN-1882E136|     Sports|-250000|
|  TXN-A20865A6|       Home|-250000|
|  TXN-B379A3D2|Electronics|-250000|
+--------------+-----------+-------+
only showing top 5 rows
--- 3. PROFILING TANGGAL (Mencari Tanggal Masa Depan) ---
+--------------+-------------------+-------+
|transaction_id|          timestamp|  price|
+--------------+-------------------+-------+
|  TXN-D3D487E7|2099-12-31 00:00:00|4888813|
|  TXN-58A26DFF|2099-12-31 00:00:00|4402499|
|  TXN-94DFC16E|2099-12-31 00:00:00|1379927|
|  TXN-F3740E94|2099-12-31 00:00:00|4030113|
|  TXN-6EB076C3|2099-12-31 00:00:00|3124769|
+--------------+-------------------+-------+
only showing top 5 rows
--- 4. PREVIEW SELURUH FIELD UNTUK MISSING VALUES (Null) ---
+--------------+-----------+---------

# **Cell 3: Proses Data Cleansing**
memperbaiki 4 jenis anomali (veracity check) yang sebelumnya kita buat. Perhatikan bagaimana fungsi-fungsi PySpark seperti dropna, withColumn, dan filter bekerja.

In [6]:
# CELL 3
from pyspark.sql.functions import col, when, lower, expr

print("Memulai proses Data Cleansing...")

# 1. Duplicate Records: Hapus duplikat berdasarkan transaction_id
df_clean = df_raw.dropDuplicates(subset=["transaction_id"])

# 2. Missing Values: Hapus baris yang memiliki nilai null pada kolom krusial
df_clean = df_clean.dropna(subset=["customer_id", "timestamp", "category"])

# 3. Inconsistent Formatting: Standarisasi nama kota menjadi 'Jakarta'
df_clean = df_clean.withColumn(
    "city",
    when(lower(col("city")).isin(["jkt", "jakarta", "dki jakarta"]), "Jakarta")
    .otherwise(col("city"))
)

# 4. Outliers & Invalid Values:
# Gunakan expr("TRY_CAST(...)") agar teks seperti "Transfer" aman diubah menjadi NULL
df_clean = df_clean.withColumn("price", expr("TRY_CAST(price AS DOUBLE)"))

# Hapus harga negatif, NULL hasil cast, dan tanggal dari masa depan
df_clean = df_clean.filter(
    (col("price").isNotNull()) &
    (col("price") >= 0) &
    (col("timestamp") < "2030-01-01")
)

print(f"Total baris setelah cleansing: {df_clean.count():,}")

Memulai proses Data Cleansing...
Total baris setelah cleansing: 899,888


In [7]:
# Bukti Data Telah Bersih

print("1. Sampel 5 Baris Data Bersih:")
df_clean.show(5)

print("2. Bukti Konsistensi Kota (Tidak ada lagi 'jkt' atau 'DKI Jakarta'):")
# Akan terlihat hanya ada 'Jakarta' murni dan kota-kota valid lainnya
df_clean.groupBy("city").count().orderBy(col("count").desc()).show()

print("3. Bukti Harga Valid (Nilai 'min' pada price tidak boleh minus):")
# Menampilkan ringkasan statistik untuk kolom price
df_clean.describe("price").show()

print("4. Bukti Tanggal Valid (Tidak ada tahun 2099):")
# Mengurutkan dari tanggal paling baru (descending) untuk melihat batas atasnya
df_clean.orderBy(col("timestamp").desc()).select("transaction_id", "timestamp").show(5)

1. Sampel 5 Baris Data Bersih:
+--------------+-----------+-------------------+----------+--------+--------+---------+--------+--------------+--------+
|transaction_id|customer_id|          timestamp|product_id|category|quantity|    price|discount|payment_method|    city|
+--------------+-----------+-------------------+----------+--------+--------+---------+--------+--------------+--------+
|  TXN-0000088D| CUST-00610|2024-01-19 05:48:00|   PRD-288| Apparel|      14|2025109.0|    0.41|           COD| Jakarta|
|  TXN-00005698| CUST-41065|2024-10-17 01:28:00|   PRD-335| Apparel|       9|2307442.0|    0.02|      Transfer|   Medan|
|  TXN-0001C3D4| CUST-42009|2023-03-12 01:18:00|   PRD-108|  Beauty|       1|4093798.0|    0.29|      Transfer|   Medan|
|  TXN-0002F92F| CUST-23269|2024-09-09 02:55:00|   PRD-178|    Home|       8|3148373.0|     0.4|   Credit Card|Surabaya|
|  TXN-00072A96| CUST-35851|2024-05-27 01:16:00|   PRD-169|  Beauty|       2|1688300.0|    0.11|   Credit Card| Jakarta|
+

# **Cell 4: Storage Benchmark (Write / Simpan Data)**
memenuhi syarat Milestone 1 Part 3. Kita akan menyimpan data yang sudah bersih ke dalam format CSV, JSONL, dan Parquet untuk melihat ukuran file-nya di penyimpanan (storage).

In [8]:
import os
print("JAVA_HOME:", os.environ.get("JAVA_HOME"))
print("HADOOP_HOME:", os.environ.get("HADOOP_HOME"))

JAVA_HOME: C:\Program Files\Java\jdk-17.0.20.1
HADOOP_HOME: C:\hadoop


In [9]:
# CELL 4
output_dir = "C:/tmp/data_curated2"

print("Menyimpan ke format CSV...")
df_clean.coalesce(1).write.mode("overwrite").csv(f"{output_dir}/transactions_clean.csv", header=True)

print("Menyimpan ke format JSONL...")
df_clean.coalesce(1).write.mode("overwrite").json(f"{output_dir}/transactions_clean.json")

print("Menyimpan ke format Parquet...")
df_clean.coalesce(1).write.mode("overwrite").parquet(f"{output_dir}/transactions_clean.parquet")

print("Penyimpanan selesai!")

Menyimpan ke format CSV...
Menyimpan ke format JSONL...
Menyimpan ke format Parquet...
Penyimpanan selesai!


# **Cell 5: Read Query Benchmark (Uji Kecepatan Baca)**
Langkah terakhir adalah mengukur berapa detik yang dibutuhkan sistem untuk membaca masing-masing format dan melakukan sedikit perhitungan data. Ini akan membuktikan mengapa format Parquet adalah yang terbaik untuk Big Data

In [10]:
# CELL 5
def benchmark_read(format_type, path):
    start_time = time.time()

    # Membaca file
    if format_type == "csv":
        df = spark.read.csv(path, header=True)
    elif format_type == "json":
        df = spark.read.json(path)
    elif format_type == "parquet":
        df = spark.read.parquet(path)

    # Memaksa eksekusi query (action) agar waktu baca akurat
    total_rows = df.count()
    end_time = time.time()

    duration = end_time - start_time
    print(f"Format {format_type.upper()} | Waktu baca: {duration:.4f} detik | Total Baris: {total_rows:,}")

print("=== HASIL BENCHMARK KECEPATAN BACA ===")
benchmark_read("csv", f"{output_dir}/transactions_clean.csv")
benchmark_read("json", f"{output_dir}/transactions_clean.json")
benchmark_read("parquet", f"{output_dir}/transactions_clean.parquet")

=== HASIL BENCHMARK KECEPATAN BACA ===
Format CSV | Waktu baca: 0.5392 detik | Total Baris: 899,888
Format JSON | Waktu baca: 1.8268 detik | Total Baris: 899,888
Format PARQUET | Waktu baca: 0.4428 detik | Total Baris: 899,888
